In [46]:
# --- Setup: imports and loading the data ---
import pandas as pd
import numpy as np

df = pd.read_parquet("statcast_2026.parquet")

print(f"Loaded {len(df):,} pitches")
print(f"Date range: {df['game_date'].min()} to {df['game_date'].max()}")

Loaded 558,489 pitches
Date range: 2026-03-25 00:00:00 to 2026-08-18 00:00:00


In [47]:
# Load the batter archetype probabilities exported from Phase 2
batter_archetypes = pd.read_parquet("batter_archetype_probs.parquet")

print(f"Loaded archetype probabilities for {len(batter_archetypes)} batters")
batter_archetypes.head()

Loaded archetype probabilities for 424 batters


,archetype_0_prob,archetype_1_prob,archetype_2_prob,archetype_3_prob,archetype_4_prob,archetype_5_prob,batter,top_archetype,top_prob
0,4.361442e-17,9.975499e-01,0.000041,1.984737e-13,3.867686e-25,2.409329e-03,500743,archetype_1_prob,0.997550
1,2.717094e-04,3.998691e-19,0.000057,9.995882e-01,5.237625e-07,8.287315e-05,502671,archetype_3_prob,0.999588
2,2.773601e-16,1.007138e-03,0.998991,1.054448e-06,4.037799e-16,7.196117e-07,514888,archetype_2_prob,0.998991
3,2.682059e-07,4.685141e-16,0.001258,9.987402e-01,1.723346e-08,1.547028e-06,516782,archetype_3_prob,0.998740
4,1.197219e-01,7.939348e-16,0.000002,3.193074e-07,2.313538e-12,8.802762e-01,518692,archetype_5_prob,0.880276


In [48]:
# Merge each pitch with the batter's archetype probabilities. Since this
# uses 'batter' as the join key, every pitch a given batter saw gets the
# same 6 probability values attached to it.
df_with_archetypes = df.merge(
    batter_archetypes[['batter', 'archetype_0_prob', 'archetype_1_prob',
                        'archetype_2_prob', 'archetype_3_prob',
                        'archetype_4_prob', 'archetype_5_prob']],
    on='batter', how='inner'  # inner join: only keep pitches to batters who made it through Phase 2's 100+ PA filter
)

print(f"Original pitch count: {len(df):,}")
print(f"Pitch count after joining to qualified batters only: {len(df_with_archetypes):,}")
print(f"({len(df_with_archetypes) / len(df):.1%} of all pitches were thrown to a qualified batter)")

Original pitch count: 558,489
Pitch count after joining to qualified batters only: 526,853
(94.3% of all pitches were thrown to a qualified batter)


In [49]:
# Reshape from wide (6 probability columns) to long (one row per
# pitch-archetype pair) so we can group by archetype like any other column
archetype_cols = [f'archetype_{i}_prob' for i in range(6)]

long_df = df_with_archetypes.melt(
    id_vars=['pitcher', 'pitch_type', 'stand', 'delta_run_exp'],
    value_vars=archetype_cols,
    var_name='archetype',
    value_name='weight'
)

# Each pitch's contribution to an archetype's average is its run value
# TIMES that batter's probability of belonging to that archetype --
# a pitch to a 90%-confident batter counts almost fully; a pitch to a
# 20%-confident batter barely counts at all toward that archetype
long_df['weighted_value'] = long_df['delta_run_exp'] * long_df['weight']

# Group by pitcher + pitch type + handedness + archetype, summing both
# the weighted values and the weights themselves -- dividing gives us
# the correct weighted average (not a simple mean)
archetype_performance = (
    long_df.groupby(['pitcher', 'pitch_type', 'stand', 'archetype'])
    .agg(weighted_sum=('weighted_value', 'sum'), weight_sum=('weight', 'sum'))
    .reset_index()
)
archetype_performance['avg_run_value'] = (
    archetype_performance['weighted_sum'] / archetype_performance['weight_sum']
)

print(f"Rows in archetype_performance: {len(archetype_performance):,}")
archetype_performance.head(10)

Rows in archetype_performance: 41,952


,pitcher,pitch_type,stand,archetype,weighted_sum,weight_sum,avg_run_value
0,434378,CH,L,archetype_0_prob,-0.022379,7.390755e-02,-0.302797
1,434378,CH,L,archetype_1_prob,0.006,1.999999e+00,0.003
2,434378,CH,L,archetype_2_prob,-0.000058,1.780308e-04,-0.32407
3,434378,CH,L,archetype_3_prob,-0.302941,9.321287e-01,-0.324999
4,434378,CH,L,archetype_4_prob,-0.0,3.710605e-08,-0.324875
5,434378,CH,L,archetype_5_prob,-0.065622,9.937868e-01,-0.066032
6,434378,CH,R,archetype_0_prob,0.0,2.054863e-09,0.000015
7,434378,CH,R,archetype_1_prob,0.0,3.421913e-08,0.022832
8,434378,CH,R,archetype_2_prob,0.022838,9.929434e-01,0.023
9,434378,CH,R,archetype_3_prob,0.000153,6.640855e-03,0.023


In [50]:
# League-wide average run value per (pitch_type, archetype), pooled
# across ALL pitchers -- this becomes the shrinkage target. Same weighted
# math as before, just grouped more broadly (no pitcher, no handedness).
league_by_archetype = (
    long_df.groupby(['pitch_type', 'archetype'])
    .agg(league_weighted_sum=('weighted_value', 'sum'), league_weight_sum=('weight', 'sum'))
    .reset_index()
)
league_by_archetype['league_avg_run_value'] = (
    league_by_archetype['league_weighted_sum'] / league_by_archetype['league_weight_sum']
)

league_by_archetype[['pitch_type', 'archetype', 'league_avg_run_value']].head(10)

,pitch_type,archetype,league_avg_run_value
0,CH,archetype_0_prob,0.007951
1,CH,archetype_1_prob,-0.004386
2,CH,archetype_2_prob,-0.004027
3,CH,archetype_3_prob,-0.005528
4,CH,archetype_4_prob,-0.001192
5,CH,archetype_5_prob,0.004065
6,CS,archetype_0_prob,-0.028929
7,CS,archetype_1_prob,0.013848
8,CS,archetype_2_prob,-0.018666
9,CS,archetype_3_prob,0.000539


In [51]:
# Merge league-level averages onto the per-pitcher archetype_performance table
archetype_performance = archetype_performance.merge(
    league_by_archetype[['pitch_type', 'archetype', 'league_avg_run_value']],
    on=['pitch_type', 'archetype'], how='left'
)

# Apply shrinkage: pitcher's own (weighted) average blended with the
# league-wide average for that pitch type + archetype, weighted by how
# much real evidence (weight_sum) backs the pitcher's own number
K = 30  # same trust level as Phase 3's shrinkage -- treat league avg as ~30 pitches of evidence

archetype_performance['avg_run_value_shrunk'] = (
    (archetype_performance['weight_sum'] * archetype_performance['avg_run_value'] +
     K * archetype_performance['league_avg_run_value'])
    / (archetype_performance['weight_sum'] + K)
)

archetype_performance[['pitcher', 'pitch_type', 'stand', 'archetype',
                        'weight_sum', 'avg_run_value', 'league_avg_run_value',
                        'avg_run_value_shrunk']].head(10)

,pitcher,pitch_type,stand,archetype,weight_sum,avg_run_value,league_avg_run_value,avg_run_value_shrunk
0,434378,CH,L,archetype_0_prob,7.390755e-02,-0.302797,0.007951,0.007187
1,434378,CH,L,archetype_1_prob,1.999999e+00,0.003,-0.004386,-0.003925
2,434378,CH,L,archetype_2_prob,1.780308e-04,-0.32407,-0.004027,-0.004029
3,434378,CH,L,archetype_3_prob,9.321287e-01,-0.324999,-0.005528,-0.015155
4,434378,CH,L,archetype_4_prob,3.710605e-08,-0.324875,-0.001192,-0.001192
5,434378,CH,L,archetype_5_prob,9.937868e-01,-0.066032,0.004065,0.001817
6,434378,CH,R,archetype_0_prob,2.054863e-09,0.000015,0.007951,0.007951
7,434378,CH,R,archetype_1_prob,3.421913e-08,0.022832,-0.004386,-0.004386
8,434378,CH,R,archetype_2_prob,9.929434e-01,0.023,-0.004027,-0.003161
9,434378,CH,R,archetype_3_prob,6.640855e-03,0.023,-0.005528,-0.005521


## Validating archetype-level personalization

Comparing the new archetype-weighted combo scores against the
handedness-only scores from Phase 3, to see how many pitchers actually
get a different recommendation once archetype is factored in. A big
shift means the extra personalization is worth it; a small shift is
still a useful, honest finding about how much the current data can
actually support.

In [52]:
all_combos = pd.read_parquet("combos_handedness_only.parquet")
print(f"Loaded {len(all_combos)} combo rows")
all_combos.head()

Loaded 2103 combo rows


,pitcher,pitch_1,pitch_2,pitch_3,combo_score_vs_L_weighted,combo_score_vs_R_weighted
0,453286,CH,CU,FF,0.007149,-0.013674
1,453286,CH,CU,SL,0.016626,0.024836
2,453286,CH,FF,SL,0.013088,-0.000338
3,453286,CU,FF,SL,0.018017,0.002977
4,500779,CH,CU,FF,-0.003621,0.003947


In [53]:
# Step 1: score each combo per archetype, same weighted-average logic
# as Phase 3's combo scoring, but now pulling from archetype_performance
# (which has 6 archetype-specific values per pitch) instead of a single
# handedness-only value.

def score_combo_by_archetype(row, stand, archetype_col):
    pitcher_id = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]

    weighted_sum = 0
    weight_total = 0
    for pt in pitches:
        match = archetype_performance[
            (archetype_performance['pitcher'] == pitcher_id) &
            (archetype_performance['pitch_type'] == pt) &
            (archetype_performance['stand'] == stand) &
            (archetype_performance['archetype'] == archetype_col)
        ]
        if not match.empty:
            value = match['avg_run_value_shrunk'].values[0]
            weight = match['weight_sum'].values[0]
            weighted_sum += value * weight
            weight_total += weight

    return weighted_sum / weight_total if weight_total > 0 else np.nan

# Score every combo against EVERY archetype, vs lefties (start with just
# lefties for now to keep this manageable -- we'll add righties after
# confirming this works)
archetype_cols = [f'archetype_{i}_prob' for i in range(6)]
for a_col in archetype_cols:
    all_combos[f'{a_col}_vs_L'] = all_combos.apply(
        lambda r: score_combo_by_archetype(r, 'L', a_col), axis=1
    )

all_combos.head()

,pitcher,pitch_1,pitch_2,pitch_3,combo_score_vs_L_weighted,combo_score_vs_R_weighted,archetype_0_prob_vs_L,archetype_1_prob_vs_L,archetype_2_prob_vs_L,archetype_3_prob_vs_L,archetype_4_prob_vs_L,archetype_5_prob_vs_L
0,453286,CH,CU,FF,0.007149,-0.013674,0.023099,0.005109,-0.001304,0.005745,-0.003882,0.001244
1,453286,CH,CU,SL,0.016626,0.024836,0.026577,-0.003711,0.010866,0.000434,0.003886,0.002945
2,453286,CH,FF,SL,0.013088,-0.000338,0.035504,-0.006529,0.004217,0.003682,-0.002924,-0.003805
3,453286,CU,FF,SL,0.018017,0.002977,0.040589,0.000020,0.007219,-0.001886,0.004369,-0.001770
4,500779,CH,CU,FF,-0.003621,0.003947,0.004906,-0.001302,-0.000763,-0.006794,-0.005399,0.008759


In [54]:
def score_combo_by_archetype_fast(row, stand, archetype_col):
    pitcher_id = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]

    weighted_sum = 0
    weight_total = 0
    for pt in pitches:
        key = (pitcher_id, pt, stand, archetype_col)
        if key in lookup:
            value, weight = lookup[key]
            weighted_sum += value * weight
            weight_total += weight

    return weighted_sum / weight_total if weight_total > 0 else np.nan

In [55]:
# For each pitcher, find their best combo for EACH archetype (vs lefties),
# then compare against their handedness-only best combo from Phase 3

results = []
for a_col in archetype_cols:
    best_per_archetype = (
        all_combos.sort_values(f'{a_col}_vs_L')
        .groupby('pitcher').first()
        .reset_index()[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3']]
    )
    best_per_archetype['archetype'] = a_col
    results.append(best_per_archetype)

archetype_best_combos = pd.concat(results, ignore_index=True)

# Compare against the handedness-only best combo
handedness_only_best = (
    all_combos.sort_values('combo_score_vs_L_weighted')
    .groupby('pitcher').first()
    .reset_index()[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3']]
    .rename(columns={'pitch_1': 'h_pitch_1', 'pitch_2': 'h_pitch_2', 'pitch_3': 'h_pitch_3'})
)

comparison = archetype_best_combos.merge(handedness_only_best, on='pitcher')

def combo_changed(row):
    archetype_set = {row['pitch_1'], row['pitch_2'], row['pitch_3']}
    handedness_set = {row['h_pitch_1'], row['h_pitch_2'], row['h_pitch_3']}
    return archetype_set != handedness_set

comparison['changed'] = comparison.apply(combo_changed, axis=1)

# Summary: what % of (pitcher, archetype) pairs get a different combo
# than the handedness-only recommendation?
summary = comparison.groupby('archetype')['changed'].agg(['sum', 'count', 'mean'])
summary['mean'] = (summary['mean'] * 100).round(1)
summary.columns = ['changed_count', 'total_pitchers', 'pct_changed']
print(summary)

                  changed_count  total_pitchers  pct_changed
archetype                                                   
archetype_0_prob            118             333         35.4
archetype_1_prob            154             333         46.2
archetype_2_prob            138             333         41.4
archetype_3_prob            151             333         45.3
archetype_4_prob            143             333         42.9
archetype_5_prob            129             333         38.7


In [56]:
lookup = {}
for row in archetype_performance.itertuples():
    key = (row.pitcher, row.pitch_type, row.stand, row.archetype)
    lookup[key] = (row.avg_run_value_shrunk, row.weight_sum)

print(f"Built lookup with {len(lookup):,} entries")

Built lookup with 41,952 entries


In [57]:
for a_col in archetype_cols:
    all_combos[f'{a_col}_vs_R'] = all_combos.apply(
        lambda r: score_combo_by_archetype_fast(r, 'R', a_col), axis=1
    )

print("Done scoring vs righties")

Done scoring vs righties


## The Best Combo per Archetype x Handedness for Each Pitcher Table

In [58]:
# Reshape the wide table (12 separate score columns: 6 archetypes x 2
# handedness) into a tidy long format: one row per (pitcher, combo,
# handedness, archetype, score) -- much easier to rank and read.

records = []
for a_col in archetype_cols:
    for stand, suffix in [('L', '_vs_L'), ('R', '_vs_R')]:
        score_col = f'{a_col}{suffix}'
        temp = all_combos[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3', score_col]].copy()
        temp['stand'] = stand
        temp['archetype'] = a_col
        temp = temp.rename(columns={score_col: 'score'})
        records.append(temp)

long_scores = pd.concat(records, ignore_index=True)

# For each (pitcher, handedness, archetype), find the single best combo --
# lowest score = best for the pitcher, same convention as always
best_combos_final = (
    long_scores.dropna(subset=['score'])
    .sort_values('score')
    .groupby(['pitcher', 'stand', 'archetype'])
    .first()
    .reset_index()
)

# Clean up archetype labels for readability
best_combos_final['archetype'] = best_combos_final['archetype'].str.replace('_prob', '')

print(f"Total (pitcher, handedness, archetype) recommendations: {len(best_combos_final):,}")
print(f"Unique pitchers covered: {best_combos_final['pitcher'].nunique()}")
best_combos_final.head(15)

Total (pitcher, handedness, archetype) recommendations: 3,996
Unique pitchers covered: 333


,pitcher,stand,archetype,pitch_1,pitch_2,pitch_3,score
0,453286,L,archetype_0,CH,CU,FF,0.023099
1,453286,L,archetype_1,CH,FF,SL,-0.006529
2,453286,L,archetype_2,CH,CU,FF,-0.001304
3,453286,L,archetype_3,CU,FF,SL,-0.001886
4,453286,L,archetype_4,CH,CU,FF,-0.003882
5,453286,L,archetype_5,CH,FF,SL,-0.003805
6,453286,R,archetype_0,CH,FF,SL,-0.000508
7,453286,R,archetype_1,CH,CU,FF,-0.008606
8,453286,R,archetype_2,CH,FF,SL,-0.022102
9,453286,R,archetype_3,CH,CU,FF,0.006334


In [59]:
# Save the final Phase 4 deliverable -- this is what the dashboard will read from
best_combos_final.to_parquet("phase4_combo_recommendations.parquet", index=False)
print(f"Saved {len(best_combos_final)} rows to phase4_combo_recommendations.parquet")

Saved 3996 rows to phase4_combo_recommendations.parquet
